# FASTA 解析
读取两个 `plus_rev.fasta` 文件，删除以 `rev_` 开头的反向序列，并分别保存为 Excel。

In [19]:
# Cell 1：导入库并设置路径
import os
import re
import pandas as pd

project_dir = "./"
raw_dir = os.path.join(project_dir, "raw_data")
output_dir = os.path.join(project_dir, "outputs", "fasta")
os.makedirs(output_dir, exist_ok=True)

In [20]:
# Cell 2：定义 FASTA 解析函数
def parse_fasta(fasta_file):
    records, header, sequence, reverse_count = [], None, [], 0

    def add_record(header, sequence):
        nonlocal reverse_count
        if header is None:
            return
        if header.startswith("rev_"):
            reverse_count += 1
            return
        first, _, details = header.partition(" ")
        database, uniprot_id, entry_name = first.split("|", 2)
        gene = re.search(r"(?:^| )GN=([^ ]+)", details)
        organism = re.search(r"(?:^| )OS=(.+?)(?= OX=| GN=| PE=| SV=|$)", details)
        description = details.split(" OS=", 1)[0]
        protein_sequence = "".join(sequence).replace(" ", "").upper()
        records.append({"UNIPROTID": uniprot_id, "FORWARD": database, 
                        "ENTRYNAME": entry_name, "GENENAME": gene.group(1) if gene else None, 
                        "PROTEINNAME": description, "ORGANISM": organism.group(1) if organism else None, 
                        "SEQUENCE": protein_sequence, "LENGTH": len(protein_sequence)})

    with open(fasta_file, encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line.startswith(">"):
                add_record(header, sequence)
                header, sequence = line[1:], []
            elif line:
                sequence.append(line)
    add_record(header, sequence)
    return pd.DataFrame(records), reverse_count

In [21]:
# Cell 3：解析两个 FASTA 文件
human, human_reverse = parse_fasta(os.path.join(raw_dir, "HS0-2_plus_rev.fasta"))
sa59, sa59_reverse = parse_fasta(os.path.join(raw_dir, "SA59_plus_rev.fasta"))
print(f"HS0-2：保留 {len(human):,} 条正向序列，排除 {human_reverse:,} 条反向序列")
print(f"SA59：保留 {len(sa59):,} 条正向序列，排除 {sa59_reverse:,} 条反向序列")

HS0-2：保留 20,868 条正向序列，排除 20,868 条反向序列
SA59：保留 2,959 条正向序列，排除 2,959 条反向序列


In [22]:
# Cell 4：检查结果
display(human.head(3))
display(sa59.head(3))

,UNIPROTID,FORWARD,ENTRYNAME,GENENAME,PROTEINNAME,ORGANISM,SEQUENCE,LENGTH
0,A0A024R161,tr,A0A024R161_HUMAN,DNAJC25-GNG10,Guanine nucleotide-binding protein subunit gamma,Homo sapiens,MGAPLLSPGWGAGAAGRRWWMLLAPLLPALLLVRPAGALVEGLYCG...,153
1,A0A024R1R8,tr,A0A024R1R8_HUMAN,hCG_2014768,"HCG2014768, isoform CRA_a",Homo sapiens,MSSHEGGKKKALKQPKKQAKEMDEEEKAFKQKQKEEQKKLEVLKAK...,64
2,A0A024RBG1,sp,NUD4B_HUMAN,NUDT4B,Diphosphoinositol polyphosphate phosphohydrola...,Homo sapiens,MMKFKPNQTRTYDREGFKKRAACLCFRSEQEDEVLLVSSSRYPDQW...,181


,UNIPROTID,FORWARD,ENTRYNAME,GENENAME,PROTEINNAME,ORGANISM,SEQUENCE,LENGTH
0,Q9F0R1,sp,SARR_STAA8,sarR,HTH-type transcriptional regulator SarR,Staphylococcus aureus (strain NCTC 8325),MSKINDINDLVNATFQVKKFFRDTKKKFNLNYEEIYILNHILRSES...,115
1,Q2FZ08,sp,RNY_STAA8,rny,Ribonuclease Y,Staphylococcus aureus (strain NCTC 8325),MNLLSLLLILLGIILGVVGGYVVARNLLLQKQSQARQTAEDIVNQA...,519
2,P69848,sp,MOAA_STAA8,moaA,"GTP 3',8-cyclase",Staphylococcus aureus (strain NCTC 8325),MVEQIKDKLGRPIRDLRLSVTDRCNFRCDYCMPKEVFGDDFVFLPK...,340


In [23]:
# Cell 5：保存为 Excel
human_file = os.path.join(output_dir, "Huamn.xlsx")
sa59_file = os.path.join(output_dir, "SA59.xlsx")
human.to_excel(human_file, index=False)
sa59.to_excel(sa59_file, index=False)
print(f"已保存：{human_file}")
print(f"已保存：{sa59_file}")

已保存：./outputs\fasta\Huamn.xlsx
已保存：./outputs\fasta\SA59.xlsx


In [24]:
print(len(human))
print(len(sa59))

20868
2959
